# Step 05 — Mock Video Evaluation

Runs the trained model on the mock videos (`data_sample/mock-videos/`):
- `Industrial-One.mp4` — warehouse packing/sorting station
- `madera.mp4` — wood/pallet manufacturing with forklift

Renders an annotated video with bounding boxes, action labels, and confidence scores.

Note: The mock videos use a **different camera setup** than InHARD (no IMU harness, different angles, real-world CCTV). This tests generalization.

In [ ]:
import sys
from pathlib import Path
NB_DIR = Path.cwd()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, Video

In [ ]:
from lib.paths import CHECKPOINTS_DIR, OUTPUTS_DIR, list_mock_videos

mock_videos = list_mock_videos()
print(f'Found {len(mock_videos)} mock video(s):')
for v in mock_videos:
    cap  = cv2.VideoCapture(str(v))
    fps  = cap.get(cv2.CAP_PROP_FPS)
    fc   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h    = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    print(f'  {v.name}: {w}×{h}  {fps:.0f}fps  {fc} frames  ({fc/fps:.1f}s)')

ckpts = sorted(CHECKPOINTS_DIR.glob('har_vjepa_*.pt'))
if ckpts:
    checkpoint = ckpts[-1]
    print(f'\nCheckpoint: {checkpoint.name}')
else:
    print('No checkpoint — run notebook 03 first.')

In [ ]:
# Render annotated eval video for each mock
from lib.eval_video import render_eval_video

for video_path in mock_videos:
    out_path = OUTPUTS_DIR / f'v2_eval_{video_path.stem}.mp4'
    print(f'\nRendering {video_path.name} → {out_path.name} …')
    stats = render_eval_video(
        video_path    = video_path,
        checkpoint    = checkpoint,
        output_path   = out_path,
        max_frames    = 600,
        infer_every   = 16,
        buffer_frames = 32,
        dwell_windows = 2,
        min_confidence= 0.25,
    )
    print(f'  frames processed : {stats.get("frames_processed", "?")}')  
    print(f'  detections       : {stats.get("n_detections", "?")}')  
    print(f'  output           : {out_path}')

In [ ]:
# Show thumbnail frames from each rendered video
for video_path in mock_videos:
    out_path = OUTPUTS_DIR / f'v2_eval_{video_path.stem}.mp4'
    if not out_path.is_file():
        continue
    cap = cv2.VideoCapture(str(out_path))
    fc  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    indices = np.linspace(0, fc-1, 5, dtype=int)
    fig, axes = plt.subplots(1, 5, figsize=(16, 3))
    for ax, idx in zip(axes, indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.axis('off')
        ax.set_title(f't={idx}', fontsize=8)
    cap.release()
    fig.suptitle(f'Annotated output — {video_path.name}', fontsize=11)
    plt.tight_layout()
    plt.show()

In [ ]:
# Inference statistics — confidence distribution
from lib.inference import load_model_for_inference
from lib.crop_extract import crops_for_embedding_with_meta
from lib.embeddings import extract_vjepa_embedding
from lib.har_model import load_checkpoint
import torch
import torch.nn.functional as F

model, info = load_checkpoint(checkpoint)
class_names = info['class_names']

for video_path in mock_videos[:1]:  # run on first mock
    meta = crops_for_embedding_with_meta(video_path, mode='full_clip', max_read=64)
    crops = meta['crops']
    if len(crops) < 4: continue
    emb = extract_vjepa_embedding(crops)
    with torch.inference_mode():
        logits = model(torch.from_numpy(emb).float().unsqueeze(0))
        probs  = F.softmax(logits, dim=-1)[0].cpu().numpy()

    fig, ax = plt.subplots(figsize=(10, 4))
    colors = ['#4CAF50' if p == probs.max() else '#2196F3' for p in probs]
    ax.barh(class_names, probs, color=colors)
    ax.axvline(0.25, color='orange', ls='--', lw=1.5, label='min_confidence=0.25')
    ax.set_xlabel('Softmax probability')
    ax.set_title(f'Inference on {video_path.name} (single clip)')
    ax.legend()
    plt.tight_layout()
    plt.show()
    top_class = class_names[probs.argmax()]
    print(f'Top prediction: {top_class} ({probs.max():.1%})')